In [4]:
# Layer 3 — Star Schema Loading
# Pipeline: LaDe Delivery Intelligence
# Script: 03_load_star_schema.ipyn
# Purpose: Load clean Parquet data into MySQL star schema

import pandas as pd
import mysql.connector
from sqlalchemy import create_engine
import urllib.parse
from datetime import datetime

In [6]:
# Database connection
CONFIG = {
    "db_host": "localhost",
    "db_user": "root",
    "db_password": "password",
    "db_name": "lade_pipeline"
}

password = urllib.parse.quote_plus(CONFIG['db_password'])
engine = create_engine(
    f"mysql+mysqlconnector://{CONFIG['db_user']}:{password}@{CONFIG['db_host']}/{CONFIG['db_name']}"
)

print("Connected to MySQL successfully")

Connected to MySQL successfully


In [12]:
#Read clean Parquet from PySpark output
print("Reading clean Parquet data...")
df = pd.read_parquet('C:\\Users\\Rishitha\\OneDrive\\Desktop\\ds-projects\\lade_staging_clean')
print(f"Rows loaded: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(f"Cities: {df['city'].value_counts().to_dict()}")
print(f"Valid records: {df['is_valid'].sum()}")

Reading clean Parquet data...
Rows loaded: 472419
Columns: ['order_id', 'from_dipan_id', 'delivery_user_id', 'poi_lng', 'poi_lat', 'aoi_id', 'typecode', 'receipt_lng', 'receipt_lat', 'delivery_date', 'receipt_timestamp', 'sign_timestamp', 'delivery_duration_minutes', 'is_valid', 'processed_at', 'pipeline_version', 'source_file', 'city']
Cities: {'Hangzhou': 185589, 'Shanghai': 165583, 'Chongqing': 121247}
Valid records: 465384


In [14]:
#Populate dim_city
city_data = pd.DataFrame({
    'city_name': ['Shanghai', 'Hangzhou', 'Chongqing'],
    'city_name_chinese': ['上海市', '杭州市', '重庆市'],
    'city_size': ['Large', 'Large', 'Large']
})

city_data.to_sql('dim_city', engine, if_exists='append', index=False)
print("dim_city populated:")
print(pd.read_sql("SELECT * FROM dim_city", engine))

dim_city populated:
   city_id  city_name city_name_chinese city_size
0        1   Shanghai               上海市     Large
1        2   Hangzhou               杭州市     Large
2        3  Chongqing               重庆市     Large


In [16]:
# Populate dim_date
dates = df['delivery_date'].dt.date.unique()
dates_df = pd.DataFrame({'full_date': dates})
dates_df['full_date'] = pd.to_datetime(dates_df['full_date'])
dates_df['day_of_week'] = dates_df['full_date'].dt.day_name()
dates_df['day_number'] = dates_df['full_date'].dt.dayofweek + 1
dates_df['month_number'] = dates_df['full_date'].dt.month
dates_df['month_name'] = dates_df['full_date'].dt.month_name()
dates_df['week_number'] = dates_df['full_date'].dt.isocalendar().week.astype(int)
dates_df['quarter'] = dates_df['full_date'].dt.quarter
dates_df['is_weekend'] = dates_df['full_date'].dt.dayofweek >= 5

dates_df.to_sql('dim_date', engine, if_exists='append', index=False)
print(f"dim_date populated: {len(dates_df)} unique dates")
print(dates_df.head(3))

dim_date populated: 15 unique dates
   full_date day_of_week  day_number  month_number month_name  week_number  \
0 2022-03-22     Tuesday           2             3      March           12   
1 2022-03-25      Friday           5             3      March           12   
2 2022-03-29     Tuesday           2             3      March           13   

   quarter  is_weekend  
0        1       False  
1        1       False  
2        1       False  


In [18]:
print(df['delivery_date'].dt.month.value_counts().sort_index())

delivery_date
3    440944
4     31475
Name: count, dtype: int64


In [20]:
# Populate dim_courier
couriers = df[['delivery_user_id', 'city']].drop_duplicates(subset=['delivery_user_id'])
couriers = couriers.rename(columns={'delivery_user_id': 'delivery_user_id'})

couriers.to_sql('dim_courier', engine, if_exists='append', index=False)
print(f"dim_courier populated: {len(couriers)} unique couriers")
print(couriers.head(3))

dim_courier populated: 1870 unique couriers
                   delivery_user_id       city
0  2ab12ace0b9fd6aa6a8ae79f7639f049  Chongqing
1  5bc17526ad7f9a3a2b6afc5494fca7eb  Chongqing
2  806adb4e9f05a84bef5ca25a1bc4eda5  Chongqing


In [22]:
#  Populate dim_depot
# Get city_ids from dim_city
city_lookup = pd.read_sql("SELECT city_id, city_name FROM dim_city", engine)
city_lookup = dict(zip(city_lookup['city_name'], city_lookup['city_id']))

depots = df[['from_dipan_id', 'city']].drop_duplicates(subset=['from_dipan_id'])
depots['city_id'] = depots['city'].map(city_lookup)
depots = depots.drop(columns=['city'])
depots = depots.rename(columns={'from_dipan_id': 'from_dipan_id'})

depots.to_sql('dim_depot', engine, if_exists='append', index=False)
print(f"dim_depot populated: {len(depots)} unique depots")
print(depots.head(3))

dim_depot populated: 134 unique depots
                      from_dipan_id city_id
0  6f051d36eb4ec0c871e78961e6ef1ca1       3
1  b351d4b8457cc20baac70fff2971bc4f       3
2  37e976ad4abb10da92c97c6b34f7f54f       3


In [24]:
# Populate fact_deliveries
# Load lookup tables
date_lookup = pd.read_sql("SELECT date_id, full_date FROM dim_date", engine)
date_lookup['full_date'] = pd.to_datetime(date_lookup['full_date'])

courier_lookup = pd.read_sql("SELECT courier_id, delivery_user_id FROM dim_courier", engine)
city_lookup_df = pd.read_sql("SELECT city_id, city_name FROM dim_city", engine)
depot_lookup = pd.read_sql("SELECT depot_id, from_dipan_id FROM dim_depot", engine)

# Merge to get surrogate keys
fact = df.copy()
fact['delivery_date'] = pd.to_datetime(fact['delivery_date'])

fact = fact.merge(date_lookup, left_on='delivery_date', right_on='full_date', how='left')
fact = fact.merge(courier_lookup, on='delivery_user_id', how='left')
fact = fact.merge(city_lookup_df, left_on='city', right_on='city_name', how='left')
fact = fact.merge(depot_lookup, on='from_dipan_id', how='left')

# Extract hour features
fact['receipt_hour'] = pd.to_datetime(fact['receipt_timestamp']).dt.hour
fact['sign_hour'] = pd.to_datetime(fact['sign_timestamp']).dt.hour

# Select only fact table columns
fact_final = fact[[
    'order_id', 'date_id', 'courier_id', 'city_id', 'depot_id',
    'delivery_duration_minutes', 'receipt_hour', 'sign_hour', 'is_valid'
]].copy()

fact_final['processed_at'] = datetime.now()
fact_final['batch_id'] = '20260506_114114'

print(f"Loading {len(fact_final)} rows into fact_deliveries...")
fact_final.to_sql('fact_deliveries', engine, if_exists='append', index=False, chunksize=10000)
print("fact_deliveries loaded successfully")
print(fact_final.head(3))

Loading 472419 rows into fact_deliveries...
fact_deliveries loaded successfully
                           order_id  date_id  courier_id  city_id  depot_id  \
0  00009fe6915928a1f4e96ba1c3ad574a        1           1        3         1   
1  00022f0b48554b76d9b3d815c4f20e68        2           2        3         2   
2  0003393bee29b22018a21bf84808c212        1           3        3         3   

   delivery_duration_minutes  receipt_hour  sign_hour  is_valid  \
0                       52.0            14         15      True   
1                      129.0             8         10      True   
2                      601.0             8         18      True   

                processed_at         batch_id  
0 2026-05-06 20:33:23.747612  20260506_114114  
1 2026-05-06 20:33:23.747612  20260506_114114  
2 2026-05-06 20:33:23.747612  20260506_114114  


In [26]:
# Layer 3 Summary — Star Schema Complete

# dim_city: 3 cities loaded
# dim_date: 15 unique dates loaded
# dim_courier: 1,870 unique couriers loaded
# dim_depot: 134 unique depots loaded
# fact_deliveries: 472,419 rows loaded
# All foreign key relationships working
# Star schema verified with JOIN query

print("Layer 3 complete. Star schema ready for SQL analysis.")

Layer 3 complete. Star schema ready for SQL analysis.
